# Starting with rag
## Requirements :
    -langchain-community
    -PyPDF
    -PymuPDF


### loading documents

In [50]:
# Text Loader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../doc_files/notes.txt")
content = loader.load()
print(content)


[Document(metadata={'source': '../doc_files/notes.txt'}, page_content='This is a simple text file content.')]


In [51]:
# Directory Loader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import JSONLoader

dirLoader = DirectoryLoader(
    "../doc_files/",
    glob="**/*.json",
    loader_cls=JSONLoader,
    loader_kwargs={"jq_schema": ".", "text_content": False},
    show_progress=False
)
content = dirLoader.load()
content

[Document(metadata={'source': 'C:\\Users\\Vivek\\Desktop\\agentic_ai\\doc_files\\config.json', 'seq_num': 1}, page_content='{"setting": "enabled", "version": 1}')]

In [52]:
# Loading Pdf File
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../doc_files/",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    loader_kwargs={"extract_images":False, "extract_tables": "markdown"},
    use_multithreading=True
)
contents = dir_loader.load()
contents

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-08-03T16:14:20+05:30', 'source': '..\\doc_files\\SowmyaG_Resume.pdf', 'file_path': '..\\doc_files\\SowmyaG_Resume.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': 'Un-named', 'subject': '', 'keywords': '', 'moddate': '2026-08-03T16:14:20+05:30', 'trapped': '', 'modDate': "D:20260803161420+05'30'", 'creationDate': "D:20260803161420+05'30'", 'page': 0}, page_content='SOWMYA G \nSoftware Developer | Python & Django Developer | Backend Development \n+91-8147588578 | g34402284@gmail.com | github.com/SowmyaGopal12 | linkedin//Sowmya G \nSUMMARY \nComputer Science undergraduate skilled in Python, Java, and Django, with hands-on experience building full-stack style applications \nusing object-oriented design and clean UI principles. Experienced building CRUD-driven applications with structured data \nmanagement, exception handling, and input validation. Strong fou

In [53]:
# Create dummy files and write the some content 

import os

storage = {
    "notes.txt": "This is a simple text file content.",
    "config.json": '{"setting": "enabled", "version": 1.0}',
    "script.py": "print('Hello from the script!')",
}

for file, content in storage.items():
    with open(f"../doc_files/{file}", "w", encoding="utf-8") as f:
        f.write(content)


print("content Written Successfully...")

content Written Successfully...


# RAG Pipeline (From Indexing to Vector Db pipleine)
### Requirements: 
    -langchain-community(PyPDFLoader and PyMuPDF)
    -langchain.textsplitter (RecurisveCharacterTextSplitter)
    pathlib

In [54]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [55]:
"""
create a function that loads all the pdf file in the dir and 
returns the whole documents by adding corresponding
metadata fields like file_name and filetype
"""
def getPdfDocs(pdfDir):
    allDocs = []
    pobj = Path(pdfDir)
    if not pobj.is_dir():
        print(f"Dir Not Found")
        return None
    print(pobj.rglob("**/*.pdf"))
    pdf_files = pobj.rglob("**/*.pdf")
    # procces the pdf files 
    for pdf_file in pdf_files:
        print(f"Processing :{pdf_file}")
        loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()
        print(f"Loaded {len(docs)} pages")
        for doc in docs:
            doc.metadata["file_name"] = pdf_file.stem
            doc.metadata["file_type"] = pdf_file.suffix
        allDocs.extend(docs)
    return allDocs
all_docs = getPdfDocs("../doc_files/")

<generator object Path.rglob at 0x000001BC117E9250>
Processing :..\doc_files\SowmyaG_Resume.pdf
Loaded 1 pages
Processing :..\doc_files\SPL.pdf
Loaded 6 pages
Processing :..\doc_files\VivekanandNaikCV.pdf
Loaded 1 pages
Processing :..\doc_files\Vivekanand_Naik_Resume.pdf
Loaded 1 pages


In [56]:
# a text splitter function
def split_doc(docs, chunkSize=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunkSize,
        chunk_overlap = chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
        length_function =len
    )
    splitted_doc = text_splitter.split_documents(docs)
    print(f"splitted {len(docs)} Docs into {len(splitted_doc)}")
    # print(f"Content: {splitted_doc[0].page_content[:200]}")
    return splitted_doc
splitted_docs = split_doc(all_docs)
splitted_docs

splitted 9 Docs into 23


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-08-03T16:14:20+05:30', 'author': 'Un-named', 'moddate': '2026-08-03T16:14:20+05:30', 'source': '..\\doc_files\\SowmyaG_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': 'SowmyaG_Resume', 'file_type': '.pdf'}, page_content='SOWMYA G \nSoftware Developer | Python & Django Developer | Backend Development \n+91-8147588578 | g34402284@gmail.com | github.com/SowmyaGopal12 | linkedin//Sowmya G \nSUMMARY \nComputer Science undergraduate skilled in Python, Java, and Django, with hands -on experience building full-stack style applications \nusing object-oriented design and clean UI principles. Experienced building CRUD -driven applications with structured data \nmanagement, exception handling, and input validation. Strong foundation in Object -Oriented Programming, file handling, and version \ncontrol through Git and GitHub. \nTECHNICAL SKILLS \nLanguages: Pyth

# Embedding and vectordb

In [57]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
import uuid
from typing import List, Dict, Any, Tuple
import numpy as np
import os

In [58]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = None
        self.model_name = model_name
        self._load_model()
    
    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Dimension : {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model: {self.model_name} : {e}")
            raise ValueError("Model Cannot Be Loaded...")
        
    def generate_embedding(self, texts: List[str]):
        try:
            encodings = self.model.encode(texts, show_progress_bar=True)
            return encodings
        except Exception as e :
            print(e)
            
    def get_embedding_dimension(self):
        if not self.model:
            raise ValueError("Model Doesn't Exist")
        return self.model.get_embedding_dimension()

embedding_manager = EmbeddingManager()
embedding_manager

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 8s [Retry 4/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 8s [Retry 5/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD h

Model Dimension : 384


In [59]:
# Vector Store
class VectorStore:
    def __init__(self, collection_name: str = "PDF_Collection", persist_dir: str = "./vector_store"):
        self.collection_name = collection_name
        self.persist_dir = persist_dir
        self.collection = None
        self.client = None
        self._load_vectorDB()
    
    # load the colleciton and client
    def _load_vectorDB(self):
        # initialize collection and client
        os.makedirs(self.persist_dir, exist_ok=True)
        try:
            self.client = chromadb.PersistentClient(path=self.persist_dir)
            self.client.delete_collection(name=self.collection_name)
            self.collection = self.client.get_or_create_collection(
                self.collection_name, 
                configuration={
                  "hnsw": {
                      "space": "cosine"
                  }
                },
                metadata={
                    "description": "PDF files for RAG"
                    }
                )
            print(f"Vector Store initialized successfully {self.collection_name}")
            print(f"Existing Documents in collection {self.collection.count()}")
        except Exception as e:
            print(e)    
    # Add docs to vector store 
    def add_docs(self, docs: List[Any], embeddings: np.ndarray):
        # prepare data 
        doc_ids = []
        doc_contents = []
        doc_embeddings = []
        metadatas = []
        for i, (doc, embedding) in enumerate(zip(docs, embeddings)):
            # unique id for each doc
            uid = f"{uuid.uuid4().hex[:8]}_{i}"
            doc_ids.append(uid)
            
            # prepare page content
            doc_contents.append(doc.page_content)
            
            # embeddings 
            doc_embeddings.append(embedding.tolist())
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["document_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            print(f"metadata: {metadata}")
            
        # add fields to vector Store 
        try:
            self.collection.add(
                ids=doc_ids,
                documents=doc_contents,
                embeddings=doc_embeddings,
                metadatas=metadatas
            )
            print(f"Successfully added {len(docs)} into vector store...")
        except Exception as e :
            print(e)


vectorstore_manager = VectorStore()

Vector Store initialized successfully PDF_Collection
Existing Documents in collection 0


In [ ]:
# doc = Document(
#     page_content="Hello, LangChain!",
#     metadata={"source": "manual_input"}
# )
# emb = embedding_manager.generate_embedding([doc.page_content])
# emb.tolist()
# vectorstore_manager.add_docs(docs=[doc], embeddings=emb)


Batches: 100%|██████████| 1/1 [00:00<00:00, 129.98it/s]

metadata: {'source': 'manual_input', 'document_index': 0, 'content_length': 17}
Successfully added 1 into vector store...


In [61]:
# Extract the texts from page content 
texts = [doc.page_content for doc in splitted_docs]

# Generate Embedding 
embeddings = embedding_manager.generate_embedding(texts)

# store embedding and chunks into vector store 
vectorstore_manager.add_docs(all_docs, embeddings)

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.40it/s]

metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-08-03T16:14:20+05:30', 'author': 'Un-named', 'moddate': '2026-08-03T16:14:20+05:30', 'source': '..\\doc_files\\SowmyaG_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': 'SowmyaG_Resume', 'file_type': '.pdf', 'document_index': 0, 'content_length': 3274}
metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-02-20T19:24:23+05:30', 'title': 'Mock_Press_Report', 'author': 'Vivekanand Naik', 'moddate': '2026-02-20T19:24:23+05:30', 'source': '..\\doc_files\\SPL.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'file_name': 'SPL', 'file_type': '.pdf', 'document_index': 1, 'content_length': 530}
metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-02-20T19:24:23+05:30', 'title': 'Mock_Press_Report', 'author': 'Vivekanand Naik', 'moddate': '2026-02-20T19:24:23+05:30'

In [62]:
# Reterival Pipeline
class Reterival:
    def __init__(self, embedding_manager: EmbeddingManager, vector_store: VectorStore):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store
    
    def reteriveContext(self, query: str, top_k: int, threshold: float = 0.0) -> list[Dict[str, Any]]:
        # Generate the qurey embedding 
        query_embedding = self.embedding_manager.generate_embedding([query])
        
        # serach in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=query_embedding.tolist(),
                n_results=top_k
            )
            reterived_docs = []
            if results["documents"] and results["documents"][0]:
                docs = results["documents"][0]
                ids = results["ids"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                
                for i, (id_, doc, metadata, distance) in enumerate(zip(ids, docs, metadatas, distances)):
                    # convert distanbce to simlarity score (Chromadb uses cosine distance)
                    similarity_score = 1 - distance
                    if similarity_score >= threshold:
                        reterived_docs.append({
                            "id": id_,
                            "content": doc,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                print(f"Reterived {len(reterived_docs)}")
            else:
                print("no document Reterived ")
            return reterived_docs
        except Exception as e:
            print(e)

reterival = Reterival(embedding_manager, vectorstore_manager)

In [63]:
print(vectorstore_manager.collection.count())
print(vectorstore_manager.collection.configuration)

10
{'hnsw': {'space': 'cosine', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}, 'spann': None, 'embedding_function': <chromadb.api.types.DefaultEmbeddingFunction object at 0x000001BC11A3A790>}


In [64]:
reterival.reteriveContext(query="Inauguration Session", top_k=3)


Batches: 100%|██████████| 1/1 [00:00<00:00, 181.35it/s]

Reterived 3


[{'id': 'db8d1f29_6',
  'content': 'Photographs:',
  'metadata': {'page_label': '6',
   'total_pages': 6,
   'moddate': '2026-02-20T19:24:23+05:30',
   'file_name': 'SPL',
   'content_length': 12,
   'source': '..\\doc_files\\SPL.pdf',
   'page': 5,
   'creationdate': '2026-02-20T19:24:23+05:30',
   'producer': 'Microsoft® Word 2021',
   'document_index': 6,
   'title': 'Mock_Press_Report',
   'file_type': '.pdf',
   'author': 'Vivekanand Naik',
   'creator': 'Microsoft® Word 2021'},
  'similarity_score': 0.4203312397003174,
  'distance': 0.5796687602996826,
  'rank': 1},
 {'id': 'dbcab90e_0',
  'content': 'Hello, LangChain!',
  'metadata': {'source': 'manual_input',
   'content_length': 17,
   'document_index': 0},
  'similarity_score': 0.1951984167098999,
  'distance': 0.8048015832901001,
  'rank': 2},
 {'id': '0f0f4c8d_5',
  'content': 'Acknowledgment  \nThe successful organization of Srinivas Premier League – Season 8 was made \npossible through the constant support and encourageme